<div align="center">
  <img src="https://pypots.com/figs/pypots_logos/PyPOTS/logo_F5BG.svg" width="160" alt="PyPOTS logo"/>
  <h3>PyPOTS: A Python Toolbox for Data Mining on Partially-Observed Time Series</h3>
  <p>
    <a href="https://pypots.com">Website</a> ·
    <a href="https://docs.pypots.com">Docs</a> ·
    <a href="https://github.com/WenjieDu/PyPOTS">GitHub</a> ·
    <a href="https://github.com/WenjieDu/BrewPOTS">Tutorials</a>
  </p>
</div>

---


# KDD'26 Tutorial — Part I: Apply PyPOTS to Time Series Analysis

## An End-to-End Pipeline for Partially-Observed Time Series (POTS)

> This notebook accompanies the KDD 2026 tutorial **"End-to-End Learning for Partially-Observed Time Series with PyPOTS"**.
> It implements the hands-on sections **I.2 → I.4** of Part I:
>
> - **I.2** Data preparation and missingness simulation
> - **I.3** Unified model training for the five POTS tasks (imputation, forecasting, classification, clustering, anomaly detection)
> - **I.4** Evaluation, visualization, and a reproducibility checklist
>
> By the end, you will have a complete pipeline that goes from raw incomplete data to ready-to-report comparison results.

**📌 Section I.1 (POTS fundamentals and missingness mechanisms) is delivered in the accompanying slides and is not repeated here.**


---

## 0. Environment Setup

This notebook runs on a **CPU** in a few minutes, using lightweight model settings (a small model dimension, few training epochs, and early stopping). You can rerun it with larger settings for better results.

**Install the dependencies.** The cell below does it for you — it works the same on Windows, Linux, macOS, and Google Colab. If you prefer to install by hand, this folder also contains a `requirements.txt`:

```bash
pip install -r requirements.txt
```

We use the PyPOTS ecosystem stack throughout:

| Module        | Role                     | What we use it for here                              |
| ------------- | ------------------------ | ---------------------------------------------------- |
| **TSDB**      | Time Series Data Beans   | downloads and caches the raw PhysioNet-2012 dataset  |
| **BenchPOTS** | Benchmark preprocessing  | loads, standardizes, splits, and injects missingness |
| **PyGrinder** | Missingness simulation   | demonstrates the MCAR / MAR / MNAR mechanisms        |
| **PyPOTS**    | Modeling toolbox         | trains and evaluates every model                     |
| **BrewPOTS**  | Tutorials (you are here) | the recipe that ties everything together             |

**Naming conventions used in this notebook.** PyPOTS and BenchPOTS follow a few consistent naming rules. Knowing them up front makes the code much easier to read:

_Dataset dictionary keys_ (every PyPOTS model takes a plain Python dict with these keys):

| Key      | Meaning                                                                                                                            |
| -------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| `X`      | the model input, shape `[n_samples, n_steps, n_features]`. `NaN` = missing value                                                   |
| `X_ori`  | the **original** values, with the artificially-masked entries revealed. Used only as evaluation ground truth, never as model input |
| `X_pred` | the future window to be **predicted** (forecasting only)                                                                           |
| `y`      | the sample labels (here: in-hospital mortality, 0 = survived, 1 = died)                                                            |

_Model hyperparameter prefixes_ (you will see these in every model below):

| Prefix / name | Meaning                            | Examples                                                                                           |
| ------------- | ---------------------------------- | -------------------------------------------------------------------------------------------------- |
| `n_*`         | a **count** (number of something)  | `n_steps` (time steps), `n_features` (variables), `n_layers` (layers), `n_heads` (attention heads) |
| `d_*`         | a **dimension** (size of a vector) | `d_model` (the model's hidden width), `d_k` / `d_v` (key/value dim), `d_ffn` (feed-forward dim)    |
| `*_rate`      | a **fraction** in `[0, 1]`         | `rate` (missing rate), `anomaly_rate`                                                              |
| `patience`    | early-stopping patience            | stop training after this many epochs without validation improvement                                |


In [ ]:
# Run this cell first to make sure every dependency is installed.
# It is safe to run more than once, and it works on Windows, Linux, macOS, and Google Colab.

# %pip is the reliable way to install from inside a notebook: it installs into the same
# Python that is running the kernel, so the packages are importable right away.
# -q keeps the output short; remove it if you want to see the full install log.
%pip install -q pypots==1.5 benchpots pygrinder tsdb numpy pandas matplotlib scikit-learn

print("All dependencies are installed. You can run the rest of the notebook now.")


In [ ]:
import time  # wall-clock timing for the efficiency log (Section I.4)

import numpy as np  # array handling; NaN marks missing values throughout
import pandas as pd  # tabular summaries of the dataset splits
import matplotlib.pyplot as plt  # all figures in this notebook

import pypots  # the modeling toolbox (models, metrics, training)
import benchpots  # dataset preprocessing / benchmark splits
import pygrinder  # missingness simulation (MCAR / MAR / MNAR)
import tsdb  # dataset download & local cache
from pypots.utils.random import set_random_seed  # one call seeds python / numpy / torch

RANDOM_SEED = 2026
set_random_seed(RANDOM_SEED)  # make every stochastic step below reproducible

# print the pinned versions so anyone re-running knows the exact environment
print("pypots   :", pypots.__version__)
print("benchpots:", benchpots.__version__)
print("pygrinder:", pygrinder.__version__)
print("tsdb     :", tsdb.__version__)
print("random seed:", RANDOM_SEED)

---

## I.2 Data Preparation and Missingness Simulation

In this section we:

1. **Load** the PhysioNet-2012 ICU dataset with BenchPOTS (a single line; TSDB caches it on disk for you);
2. **Inspect** the standardized preprocessing (z-score normalization, train/val/test split);
3. **Demonstrate** how to inject controlled missingness with PyGrinder (MCAR / MAR / MNAR) for reproducible benchmarking.

> **Why PhysioNet-2012?** It is a real-world clinical time-series dataset with an _inherently_ high missing rate (~80%). The usual assumption that data is complete simply does not hold here, which makes it a standard POTS benchmark.


### I.2.1 Load the dataset

BenchPOTS' `preprocess_*` functions handle the full preparation pipeline for you:

- **downloading** the raw data (through TSDB);
- **standardization** of feature scales;
- **splitting** into train / validation / test sets;
- **masking**: hiding part of the observed values in the validation and test sets to serve as **ground truth** for evaluation.

The `pattern` and `rate` arguments control this _artificial_ evaluation missingness, which is added on top of the natural missingness already in the training set.


In [ ]:
from benchpots.datasets import preprocess_physionet2012

# one call downloads (via TSDB), standardizes, splits, and masks ground truth for evaluation
physionet2012 = preprocess_physionet2012(
    subset="set-a",  # 4000 ICU patients with in-hospital mortality labels
    pattern="point",  # mask individual observed points in val/test as evaluation ground truth
    rate=0.1,  # hide 10% of the *observed* values in val/test to score against
)

# inspect what the preprocessing pipeline returned
print("Dataset keys:", sorted(physionet2012.keys()))

### I.2.2 Inspect the preprocessed data

The returned dictionary follows the BenchPOTS convention:

| Key                                            | Meaning                                                                                             |
| ---------------------------------------------- | --------------------------------------------------------------------------------------------------- |
| `train_X` / `val_X` / `test_X`                 | model inputs, `[n_samples, n_steps, n_features]`, `NaN` = missing                                   |
| `val_X_ori` / `test_X_ori`                     | validation/test data **with the artificially-masked values revealed** (ground truth for evaluation) |
| `train_y` / `val_y` / `test_y`                 | sample labels (here: in-hospital mortality, binary)                                                 |
| `n_steps`, `n_features`, `n_classes`, `scaler` | metadata                                                                                            |


In [ ]:
# pull the metadata we will need to initialize every model below
n_steps = physionet2012["n_steps"]  # time steps per sample (48 hours)
n_features = physionet2012["n_features"]  # variables per step (37 vitals/labs)
n_classes = physionet2012["n_classes"]  # label classes (2: survived / died)

# tabulate split sizes and their natural missing rates side by side
summary = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "n_samples": [
            len(physionet2012["train_X"]),
            len(physionet2012["val_X"]),
            len(physionet2012["test_X"]),
        ],
        "missing_rate": [
            np.isnan(physionet2012["train_X"]).mean(),  # fraction of NaN in each split
            np.isnan(physionet2012["val_X"]).mean(),
            np.isnan(physionet2012["test_X"]).mean(),
        ],
    }
)
print(f"n_steps={n_steps}, n_features={n_features}, n_classes={n_classes}")
print(summary.to_string(index=False))

### I.2.3 Visualize a partially-observed sample

A single patient's 48-hour record, shown two ways:

- **Heatmap** (top): an overview of all 37 features at once. Blank cells are missing values, and most of the grid is blank — this is what POTS looks like in practice. This view shows the overall missing pattern.
- **Per-channel lines** (bottom): a zoom into the few most-observed features, drawn as normal time series with each missing point marked by a red cross along the bottom. This view shows the actual values and exactly where the gaps fall.


In [ ]:
sample = physionet2012["train_X"][
    0
]  # one patient; a concrete case beats aggregate stats for a first look

# ---- Overview: all 37 features as a heatmap (time on the x-axis).
# Blank cells are missing values — masked out, not plotted as zeros.
fig, ax = plt.subplots(
    figsize=(14, 3.5)
)  # wide figure: 48 time steps need horizontal room
im = ax.imshow(
    np.ma.masked_invalid(
        sample
    ).T,  # mask NaNs so missing entries render as blank, not as a color
    aspect="auto",
    cmap="viridis",
    interpolation="none",  # one cell = one raw observation, no smoothing
)
ax.set_title("All features over time (blank = missing)")
ax.set_ylabel("feature")
ax.set_xlabel("time step")
fig.colorbar(im, ax=ax, shrink=0.8)  # one shared scale: features are already normalized
plt.show()

# ---- Detail: a few features as normal time-series lines, with missing points marked.
# The heatmap shows the overall missing pattern; lines show the actual values and where the gaps fall.
n_show = 4
observation_count = (~np.isnan(sample)).sum(
    axis=0
)  # observations per feature in this patient
feature_ids = np.sort(
    np.argsort(-observation_count)[:n_show]
)  # the most-observed features, in feature order

time_steps = np.arange(sample.shape[0])
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.1 * n_show), sharex=True)
for ax, feature_id in zip(axes, feature_ids):
    series = sample[:, feature_id]
    observed = ~np.isnan(series)
    ax.plot(
        time_steps[observed],
        series[observed],
        "o-",
        color="steelblue",
        markersize=4,
        linewidth=1.2,
        label="observed",
    )
    # draw missing markers along the bottom of each panel so they never overlap the curve
    ymin, ymax = series[observed].min(), series[observed].max()
    y_miss = ymin - 0.15 * (ymax - ymin + 1e-6)
    ax.plot(
        time_steps[~observed],
        np.full((~observed).sum(), y_miss),
        "x",
        color="red",
        markersize=6,
        markeredgewidth=1.5,
        linestyle="none",
        label="missing",
    )
    ax.set_ylim(y_miss - 0.1 * (ymax - ymin + 1e-6), ymax + 0.1 * (ymax - ymin + 1e-6))
    ax.set_ylabel(f"feature {feature_id}", fontsize=9)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("time step")
fig.suptitle(f"Patient 0 — the {n_show} most-observed features (red cross = missing)")
plt.tight_layout()
plt.show()

### I.2.4 Controlled missingness simulation with PyGrinder

For reproducible benchmarking we often need to inject missingness of a **known mechanism** into (relatively) complete data, so that ground truth is available.
PyGrinder implements the three canonical mechanisms introduced in Section I.1:

- **MCAR** (Missing Completely At Random): `pygrinder.mcar(X, p)` — every value is dropped independently with probability `p`;
- **MAR** (Missing At Random): `pygrinder.mar_logistic(X, obs_rate, missing_rate)` — missingness depends on _observed_ values;
- **MNAR** (Missing Not At Random): e.g. `pygrinder.mnar_x(X, offset)` — missingness depends on the _unobserved_ values themselves;
- plus structured patterns such as `pygrinder.seq_missing` (block/subsequence missing).

We demonstrate each mechanism on one complete sequence and compare the resulting missing patterns.


In [ ]:
# Build one fully-observed sample from val_X_ori for the demonstration.
# We need complete ground truth so that every hole we see comes from the injected mechanism, not the original data.
demo = physionet2012["val_X_ori"][0].copy()
demo = demo[
    ~np.isnan(demo).all(axis=1)
]  # drop all-missing time steps (tail padding common in this dataset)
demo = np.nan_to_num(demo)  # ensure completeness for the demo
demo_3d = demo[
    np.newaxis, ...
]  # most PyGrinder funcs want [n_samples, n_steps, n_features]

# inject each mechanism once so participants can compare the mask *shapes* side by side
missing_demos = {
    "MCAR (p=0.3)": pygrinder.mcar(demo_3d, p=0.3)[
        0
    ],  # uniform holes; the benign benchmark case
    "MAR (logistic)": pygrinder.mar_logistic(
        demo, obs_rate=0.5, missing_rate=0.3
    ),  # takes 2D input; holes depend on observed values
    "MNAR (value-dep.)": pygrinder.mnar_x(demo_3d, offset=0.0)[
        0
    ],  # takes 3D; holes depend on the missing values themselves
    "Block missing": pygrinder.seq_missing(demo_3d, p=0.1, seq_len=5)[
        0
    ],  # contiguous gaps, mimics sensor outages
}

# plot masks (observed = colored) rather than values: the mechanism is the point here, not the data
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharex=True, sharey=True)
for ax, (name, arr) in zip(axes, missing_demos.items()):
    rate = np.isnan(
        arr
    ).mean()  # realized rate often differs from the nominal argument — always report it
    ax.imshow((~np.isnan(arr)).T, aspect="auto", cmap="Blues", interpolation="none")
    ax.set_title(f"{name}\nrate={rate:.2f}")
    ax.set_xlabel("time step")
axes[0].set_ylabel("feature")
plt.tight_layout()
plt.show()

### I.2.5 Assemble PyPOTS-ready dataset dictionaries

Every PyPOTS model consumes plain dictionaries of NumPy arrays. We build the three splits once and reuse them for all tasks below. Two details matter here:

- **Indicating mask**: for imputation evaluation we need a mask marking exactly which values were artificially masked out in the test set — the evaluation is computed only on those positions.
- **`X_ori` in validation**: _imputation_ models additionally expect `X_ori` in the validation set (used as the early-stopping metric target), while downstream-task models only need `X`/`y`.


In [ ]:
# mask of artificially-missing values: NaN in X but observed in X_ori.
# Only these positions have known ground truth to score against; naturally-missing entries can never be evaluated.
physionet2012["val_X_indicating_mask"] = np.isnan(physionet2012["val_X"]) ^ np.isnan(
    physionet2012["val_X_ori"]
)
physionet2012["test_X_indicating_mask"] = np.isnan(physionet2012["test_X"]) ^ np.isnan(
    physionet2012["test_X_ori"]
)
# models consume X_ori as supervision targets, and NaN targets would poison the loss — fill them (masked out at scoring time anyway)
physionet2012["val_X_ori"] = np.nan_to_num(physionet2012["val_X_ori"])
physionet2012["test_X_ori"] = np.nan_to_num(physionet2012["test_X_ori"])

# assemble the dicts PyPOTS models expect; every model in the ecosystem reads this same format
# for imputation models (validation metric is computed against X_ori)
train_set = {"X": physionet2012["train_X"], "y": physionet2012["train_y"]}
val_set_with_ori = {
    "X": physionet2012["val_X"],
    "X_ori": physionet2012[
        "val_X_ori"
    ],  # needed so val MAE is computed on artificially-missing entries during training
    "y": physionet2012["val_y"],
}
test_set = {
    "X": physionet2012["test_X"],
    "y": physionet2012["test_y"],
}  # no X_ori here: test truth must never touch the model
# downstream-task models (classification etc.) only need X / y in the val set
val_set = {"X": physionet2012["val_X"], "y": physionet2012["val_y"]}

print(
    "train:",
    train_set["X"].shape,
    "| val:",
    val_set_with_ori["X"].shape,
    "| test:",
    test_set["X"].shape,
)
print(
    "positive rate (train):",
    train_set["y"].mean().round(4),
    "→ imbalanced binary problem",
)  # warns against accuracy-only evaluation later

---

## I.3 Unified Model Training for the Five POTS Tasks

One of PyPOTS' design goals is a **consistent API across all tasks and models**:

```python
model = SomeModel(...hyperparameters...)   # 1. initialize
model.fit(train_set, val_set)              # 2. train with early stopping
results = model.predict(test_set)          # 3. inference
imputed = model.impute(dataset)            #    (imputation models only)
```

We now go through the five tasks one by one. For each model we record the **training time** and the **number of trainable parameters**, so that in Section I.4 we can compare not only accuracy but also efficiency.


In [ ]:
# Collect artifacts for the final report (Section I.4)
experiment_log = []  # each entry: dict(task, model, params, train_seconds, ...)


def log_experiment(task, model_name, model, train_seconds, metrics=None):
    # one helper for all tasks so the Section I.4 table stays consistent no matter who logs it
    # training-free models (e.g. LOCF) have no underlying torch model -> 0 parameters
    torch_model = getattr(
        model, "model", None
    )  # PyPOTS wraps the torch module in .model
    if torch_model is not None and hasattr(torch_model, "parameters"):
        n_params = sum(
            param.numel() for param in torch_model.parameters() if param.requires_grad
        )  # count trainable params only
    else:
        n_params = 0
    entry = {
        "task": task,
        "model": model_name,
        "n_params": n_params,  # model size: needed for the robustness-efficiency tradeoff discussion
        "train_seconds": round(
            train_seconds, 1
        ),  # wall-clock cost, same granularity for all entries
    }
    if metrics:
        entry.update(
            metrics
        )  # metrics stay a free-form dict so each task can log its own
    experiment_log.append(entry)
    print(
        f"[logged] {task}/{model_name}: {n_params:,} params, {train_seconds:.1f}s"
    )  # immediate feedback that the log actually grew

### I.3.1 Imputation — SAITS

**SAITS** (Self-Attention-based Imputation for Time Series, ESWA 2023) is a Transformer-style imputer. It is trained on two objectives at once, whose names appear as the weights `ORT_weight` and `MIT_weight` in the code below:

- **ORT** (Observed Reconstruction Task): reconstruct the values the model _can already see_, which teaches it the normal data distribution;
- **MIT** (Masked Imputation Task): predict values that were _deliberately hidden_ during training, which teaches it to fill gaps.

The `d_*` arguments below are dimensions, following the naming table in Section 0:

- `d_model`: the model's hidden width;
- `d_k` / `d_v`: the per-head key/value dimensions;
- `d_ffn`: the feed-forward network's inner width.


In [ ]:
from pypots.imputation import SAITS

saits = SAITS(
    n_steps=n_steps,
    n_features=n_features,
    n_layers=2,  # shallow stack: enough capacity for this dataset, keeps tutorial runtime low
    d_model=64,  # hidden width of the model (d_* = dimension; see the naming table in Section 0)
    n_heads=4,  # attention heads; each head learns a different temporal pattern
    d_k=16,  # key dimension per head
    d_v=16,  # value dimension per head
    d_ffn=128,  # inner width of the feed-forward network between attention layers
    dropout=0.1,  # mild regularization; the dataset is small
    ORT_weight=1,  # weight of the observed-reconstruction term in SAITS's joint loss
    MIT_weight=1,  # weight of the masked-imputation term; equal weights = the paper's default
    batch_size=32,
    epochs=10,  # lightweight config for the tutorial; raise to 100+ for better results
    patience=3,  # early stopping on val loss so we do not waste epochs past convergence
    num_workers=0,  # no subprocess dataloading; avoids multiprocessing issues on laptops / notebooks
    device=None,  # let PyPOTS pick the best available device (cuda / mps / cpu)
    saving_path="tutorial_results/imputation/saits",
    model_saving_strategy="best",  # keep the best-val checkpoint, not the last — protects against late overfitting
)

t0 = time.time()  # wall-clock start; feeds the efficiency log (Section I.4)
saits.fit(
    train_set, val_set_with_ori
)  # val_set_with_ori lets SAITS early-stop on imputation MAE, not just loss
saits_train_seconds = time.time() - t0

In [ ]:
saits_imputation = saits.predict(test_set)[
    "imputation"
]  # predict returns a dict; we only need the imputed series here

from pypots.nn.functional import (
    calc_mae,
    calc_mse,
    calc_mre,
)  # same metric implementations used internally by PyPOTS training

mask = physionet2012[
    "test_X_indicating_mask"
]  # only evaluate on artificially-masked positions, never on naturally-missing ones
truth = physionet2012["test_X_ori"]  # ground truth behind those artificial holes
saits_metrics = {
    "MAE": calc_mae(saits_imputation, truth, mask),
    "MSE": calc_mse(
        saits_imputation, truth, mask
    ),  # penalizes large errors more than MAE — a different robustness angle
    "MRE": calc_mre(
        saits_imputation, truth, mask
    ),  # relative error; sensitive near zero, so read it together with MAE
}
print({k: round(float(v), 4) for k, v in saits_metrics.items()})

log_experiment(
    "imputation",
    "SAITS",
    saits,
    saits_train_seconds,
    metrics={k: round(float(v), 4) for k, v in saits_metrics.items()},
)  # tensors -> plain floats for the log

A single metric such as MAE tells you _how large_ the error is, but not _which features_ cause it. Placing the three stages side by side makes per-feature failures easy to see:

- **Model input**: blanks are missing values;
- **Imputed result**: if one feature row looks wrong here, that feature was not reconstructed well;
- **Ground truth**: what the complete record actually looks like.

The overall MAE alone would not have told you any of this.


In [ ]:
sample_idx = 0
# transpose so that time runs along the x-axis in all three panels.
# GridSpec keeps the three panels aligned by giving the colorbar its own narrow column.
fig = plt.figure(figsize=(14, 7))
gs = fig.add_gridspec(3, 2, width_ratios=[1, 0.02], wspace=0.02, hspace=0.35)
axes = [fig.add_subplot(gs[r, 0]) for r in range(3)]
cax = fig.add_subplot(
    gs[:, 1]
)  # dedicated colorbar axis spanning all rows: one scale bar, no panel shrinkage
for r in range(1, 3):
    axes[r].sharex(axes[0])  # shared time axis so the three panels line up vertically

panels = [
    (
        "Model input (blank = missing)",
        np.ma.masked_invalid(test_set["X"][sample_idx]).T,
    ),  # what the model actually sees
    ("SAITS imputation", saits_imputation[sample_idx].T),  # the model's reconstruction
    ("Ground truth", truth[sample_idx].T),  # what it should have said
]
vmin, vmax = (
    truth[sample_idx].min(),
    truth[sample_idx].max(),
)  # fix the color scale on ground truth so panels are comparable
for ax, (title, arr) in zip(axes, panels):
    im = ax.imshow(arr, aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("feature")
axes[-1].set_xlabel("time step")
for ax in axes[:-1]:
    plt.setp(
        ax.get_xticklabels(), visible=False
    )  # hide redundant tick labels on shared axes; keep only the bottom one
fig.colorbar(im, cax=cax)
fig.suptitle(f"Test sample {sample_idx}: from POTS input to imputed series")
plt.show()

**Imputation quality under different missing rates.**
To study robustness, we re-mask the test inputs at increasing rates with `pygrinder.mcar` and re-evaluate _without retraining_.


In [ ]:
missing_rates = [
    0.1,
    0.3,
    0.5,
    0.7,
    0.9,
]  # stress levels from mild to extreme; 0.9 probes the model's breaking point
robustness_curve = []

set_random_seed(
    RANDOM_SEED
)  # re-seed so each run masks the same positions — the curve must be reproducible
for rate in missing_rates:
    test_X_hard = pygrinder.mcar(
        test_set["X"], p=rate
    )  # inject *extra* MCAR holes on top of the existing test missingness
    indicating = np.isnan(test_X_hard) ^ np.isnan(
        truth
    )  # positions with known truth to score against, as in Section I.3
    imputed = saits.impute(
        {"X": test_X_hard}
    )  # reuse the already-trained SAITS: no retraining, we probe robustness
    mae = float(calc_mae(imputed, truth, indicating))
    robustness_curve.append({"missing_rate": rate, "MAE": mae})
    print(
        f"missing rate {rate:.0%} → MAE {mae:.4f}"
    )  # print per-rate so degradation is visible as the loop runs

robustness_df = pd.DataFrame(
    robustness_curve
)  # tabular form for plotting and the report artifacts

### I.3.2 Forecasting — SegRNN on incomplete inputs

Forecasting models in PyPOTS expect an extra key `X_pred`: the future window to be predicted. BenchPOTS generates it for you with `task_type="forecasting"`. We use a lightweight **SegRNN** forecaster on a random-walk dataset here to keep the demo fast; the same API applies to PhysioNet-style data.

A good average MSE can still hide one badly-forecast channel, so we plot the observed history, the ground-truth future, and the forecast for _each_ feature — the dashed vertical line marks where observation ends and prediction begins. Checking every channel before trusting an average metric is a habit worth keeping.


In [ ]:
from benchpots.datasets import preprocess_random_walk
from pypots.forecasting import SegRNN

# random-walk data again, but task_type="forecasting" changes the dict: it adds X_pred windows
# holding the future steps the model must predict from the (partially observed) history
forecast_ds = preprocess_random_walk(
    n_steps=48,
    n_features=5,
    n_classes=2,
    n_samples_each_class=200,
    missing_rate=0.1,  # 10% of the history is missing — forecasting must cope with holes
    task_type="forecasting",
    n_pred_steps=8,  # forecast horizon: 8 steps ahead
    random_state=RANDOM_SEED,
)

# train/val carry both history (X) and future truth (X_pred) so the model can learn and early-stop;
# test carries only X — we hold out test_X_pred for scoring afterwards
forecast_train = {"X": forecast_ds["train_X"], "X_pred": forecast_ds["train_X_pred"]}
forecast_val = {"X": forecast_ds["val_X"], "X_pred": forecast_ds["val_X_pred"]}
forecast_test = {"X": forecast_ds["test_X"]}

segrnn = SegRNN(
    n_steps=forecast_ds["n_steps"],
    n_features=forecast_ds["n_features"],
    n_pred_steps=forecast_ds["n_pred_steps"],
    n_pred_features=forecast_ds["n_pred_features"],
    seg_len=8,  # segment length; must divide the horizon cleanly
    d_model=128,
    dropout=0.1,
    batch_size=32,
    epochs=50,
    patience=5,  # early stopping on val forecast error
    num_workers=0,
    device=None,  # let PyPOTS pick GPU if available
    saving_path="tutorial_results/forecasting/segrnn",
    model_saving_strategy="best",  # keep the val-best checkpoint, not the last epoch
)

t0 = time.time()
segrnn.fit(forecast_train, forecast_val)
segrnn_train_seconds = time.time() - t0  # timing feeds the efficiency log in I.4

In [ ]:
forecast_pred = segrnn.predict(forecast_test)[
    "forecasting"
]  # unified predict() returns a dict keyed by task name

# score against the held-out future windows; no mask needed — X_pred is fully observed on synthetic data
forecast_metrics = {
    "MAE": float(calc_mae(forecast_pred, forecast_ds["test_X_pred"])),
    "MSE": float(
        calc_mse(forecast_pred, forecast_ds["test_X_pred"])
    ),  # MSE punishes large errors more than MAE
}
print({k: round(v, 4) for k, v in forecast_metrics.items()})
log_experiment(
    "forecasting",
    "SegRNN",
    segrnn,
    segrnn_train_seconds,
    metrics={k: round(v, 4) for k, v in forecast_metrics.items()},
)

# visualize all feature forecasts for one test sample: numbers alone don't show *where* errors land
n_features_to_plot = forecast_ds["n_pred_features"]
fig, axes = plt.subplots(
    1, n_features_to_plot, figsize=(4 * n_features_to_plot, 3.5), sharex=True
)
history_len = forecast_ds["n_steps"]
for feature_id in range(n_features_to_plot):
    ax = axes[feature_id]
    history = np.nan_to_num(
        forecast_ds["test_X"][0, :, feature_id]
    )  # NaN -> 0 only for display; model saw the mask
    ax.plot(range(history_len), history, label="observed history")
    future_steps = range(history_len, history_len + forecast_ds["n_pred_steps"])
    ax.plot(
        future_steps,
        forecast_ds["test_X_pred"][0, :, feature_id],
        "o-",
        label="ground truth",
    )
    ax.plot(
        future_steps, forecast_pred[0, :, feature_id], "s--", label="SegRNN forecast"
    )
    ax.axvline(
        history_len - 1, color="gray", ls=":", alpha=0.6
    )  # observation/forecast boundary
    ax.set_title(f"feature {feature_id}")
axes[0].legend(fontsize=8)
fig.suptitle("SegRNN forecasts on every feature (test sample 0, dashed = forecast)")
plt.tight_layout()
plt.show()

### I.3.3 Classification — GRU-D

We now return to PhysioNet-2012 for in-hospital mortality prediction. We use an **end-to-end** model: it reads the partially-observed input directly and learns imputation-aware representations together with the classification objective, with no separate imputation step.

We use **GRU-D**, which extends a GRU with a _decay mechanism_:

- the longer a value has been missing, the more the model forgets the stale observation and leans on the feature's overall mean;
- this is exactly what clinical data needs — a vital that has not been measured for many hours carries less current information.

A note on expectations:

- we benchmarked the available end-to-end classifiers on this dataset (GRU-D, SAITS, BRITS), and GRU-D is both the strongest and the fastest;
- the scores may look modest, but that is the honest difficulty of this task — ~80% of the values are missing, and the end-to-end model never gets to see a clean version of the data.

One practical detail when you evaluate any PyPOTS classifier: `predict` returns both `classification` (hard 0/1 labels) and `classification_proba` (soft probabilities).

- ROC and precision–recall curves need the **soft** scores;
- if you feed the hard labels instead, the score has only two distinct values, so the curve becomes a few straight segments and the AUC is understated (0.62 with hard labels vs. 0.77 with soft probabilities on the identical model here).


In [ ]:
# ---- End-to-end: GRU-D reads the partially-observed data directly, no imputation step
from pypots.classification import GRUD
from pypots.nn.functional.classification import calc_binary_classification_metrics

grud = GRUD(
    n_steps=n_steps,
    n_features=n_features,
    n_classes=n_classes,
    rnn_hidden_size=256,  # larger hidden state: this task is hard (~80% missing), so give it capacity
    batch_size=32,
    epochs=50,  # more epochs than the other demos; GRU-D converges slowly on sparse data
    patience=8,  # wider early-stopping window for the slower convergence
    num_workers=0,
    device=None,
    saving_path="tutorial_results/classification/grud",
    model_saving_strategy="best",  # keep the val-best checkpoint, not the last
)

t0 = time.time()  # wall-clock start; feeds the efficiency log (Section I.4)
grud.fit(
    train_set, val_set
)  # same fit() signature as the imputers/forecaster — unified API
grud_train_seconds = time.time() - t0

# PyPOTS classifiers return two things: "classification" (hard 0/1 labels, from argmax)
# and "classification_proba" (soft probabilities). ROC/PR curves need the soft scores —
# with only 0/1 labels there are just two distinct score values, so the curve collapses
# to a few straight segments and the AUC is meaningless.
grud_probs = grud.predict(test_set)["classification_proba"]
grud_metrics = calc_binary_classification_metrics(grud_probs, test_set["y"])
print(
    "End-to-end GRU-D:",
    {k: round(float(grud_metrics[k]), 4) for k in ("roc_auc", "pr_auc")},
)

log_experiment(
    "classification",
    "GRU-D (end-to-end)",
    grud,
    grud_train_seconds,
    metrics={
        "ROC-AUC": round(float(grud_metrics["roc_auc"]), 4),
        "PR-AUC": round(float(grud_metrics["pr_auc"]), 4),
    },
)

A single AUC value summarizes a classifier over _all_ thresholds at once, so it hides the threshold you will actually use. The three panels give the full picture:

- **ROC curve**: the full false-positive/true-positive tradeoff;
- **Precision–recall curve**: the honest view under class imbalance — the dashed line is the no-skill baseline (with ~14% positives, a naive classifier scores 0.14 precision), so the gap above that line is the real signal;
- **Confusion matrix**: fixes one threshold (0.5) and shows the concrete counts of false alarms versus missed deaths.


In [ ]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# GRU-D returns soft probabilities for both classes (n, 2); keep only P(died) for the curves
positive_class_probs = np.asarray(grud_probs)
if positive_class_probs.ndim == 2 and positive_class_probs.shape[1] == 2:
    positive_class_probs = positive_class_probs[:, 1]
else:
    positive_class_probs = positive_class_probs.reshape(-1)
true_labels = test_set["y"]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

fpr, tpr, _ = roc_curve(true_labels, positive_class_probs)
axes[0].plot(
    fpr, tpr, color="steelblue", label=f"GRU-D (AUC = {grud_metrics['roc_auc']:.2f})"
)
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)  # diagonal = random classifier reference
axes[0].set_xlabel("false positive rate")
axes[0].set_ylabel("true positive rate")
axes[0].set_title("ROC curve")
axes[0].legend(fontsize=9)

prec, rec, _ = precision_recall_curve(true_labels, positive_class_probs)
axes[1].plot(
    rec, prec, color="darkorange", label=f"GRU-D (AUC = {grud_metrics['pr_auc']:.2f})"
)
axes[1].axhline(
    true_labels.mean(), color="k", ls="--", alpha=0.4, label="class prevalence"
)  # PR baseline = the positive rate; the gap above it is the real signal
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision–recall curve (imbalanced data)")
axes[1].legend(fontsize=9)

confusion = confusion_matrix(true_labels, (positive_class_probs >= 0.5).astype(int))
ConfusionMatrixDisplay(confusion, display_labels=["survived", "died"]).plot(
    ax=axes[2], colorbar=False
)
axes[2].set_title("Confusion matrix\nGRU-D")
plt.tight_layout()
plt.show()

### I.3.4 Clustering — CRLI

**CRLI** (Clustering Representation Learning on Incomplete time-series data) clusters partially-observed sequences end-to-end. It alternates between a generator that imputes the missing values and a discriminator that pushes the imputations to be realistic, while a k-means objective shapes the learned representation into clusters.

Clustering has no labels at inference time, so PyPOTS offers two kinds of validation:

- **External validation**: against ground-truth labels — available here because the data is synthetic;
- **Internal validation**: label-free geometry (silhouette, Calinski–Harabasz, Davies–Bouldin).

Reporting both shows how to benchmark a clusterer _and_ how to sanity-check it in the wild where no labels exist.

We chose CRLI over the alternative (VaDER) for this demo because it is far more stable on a small synthetic set:

- VaDER's variational objective can collapse to a single cluster depending on the random initialization;
- CRLI reliably recovers both clusters here.


In [ ]:
from pypots.clustering import CRLI
from pypots.nn.functional import (
    calc_external_cluster_validation_metrics,
    calc_internal_cluster_validation_metrics,
)

# load a small labeled clustering set (synthetic random walk, so ground-truth classes are known)
cluster_ds = preprocess_random_walk(
    n_steps=48,
    n_features=5,
    n_classes=2,
    n_samples_each_class=200,
    missing_rate=0.1,
    task_type="clustering",
    random_state=RANDOM_SEED,
)

# re-seed right before fitting so the demo is reproducible regardless of how the
# preceding cells consumed the RNG
set_random_seed(RANDOM_SEED)

crli = CRLI(
    n_steps=cluster_ds["n_steps"],
    n_features=cluster_ds["n_features"],
    n_clusters=cluster_ds[
        "n_classes"
    ],  # we know there are 2 classes; in practice you would sweep this
    n_generator_layers=1,  # one RNN layer in the imputation generator is enough for 5 features
    rnn_hidden_size=64,
    batch_size=32,
    epochs=10,
    patience=3,
    num_workers=0,
    device=None,
    saving_path="tutorial_results/clustering/crli",
    model_saving_strategy="best",
)

t0 = time.time()
crli.fit({"X": cluster_ds["train_X"]})  # unsupervised: no labels needed for training
crli_train_seconds = time.time() - t0

cluster_pred = crli.predict({"X": cluster_ds["test_X"]})["clustering"]

# external validation: PyPOTS-native, needs ground-truth labels (available on synthetic data)
ext = calc_external_cluster_validation_metrics(cluster_pred, cluster_ds["test_y"])
# internal validation: label-free geometry; only defined for >= 2 clusters, so guard it
n_found = len(np.unique(cluster_pred))
if n_found >= 2:
    flat_test = np.nan_to_num(cluster_ds["test_X"]).reshape(
        len(cluster_ds["test_X"]), -1
    )
    intn = calc_internal_cluster_validation_metrics(flat_test, cluster_pred)
else:
    intn = {
        "note": f"model collapsed to {n_found} cluster(s); internal indices need >= 2"
    }

print(f"clusters found: {n_found}")
print("external (vs. true labels):", {k: round(float(v), 4) for k, v in ext.items()})
print(
    "internal (label-free)     :",
    {k: (round(float(v), 4) if isinstance(v, float) else v) for k, v in intn.items()},
)

c_metrics = {"RI": float(ext["rand_index"]), "ARI": float(ext["adjusted_rand_index"])}
log_experiment(
    "clustering",
    "CRLI",
    crli,
    crli_train_seconds,
    metrics={k: round(v, 4) for k, v in c_metrics.items()},
)

Internal metrics reduce cluster quality to a single number; a 2-D projection shows whether the clusters are actually separable. The figure below does the following:

- **Projection**: each partially-observed test sequence (with NaN filled) is projected down to 2-D using t-SNE;
- **Left panel**: the projection colored by the true label;
- **Right panel**: the same projection colored by CRLI's assignment.

If the two panels show the same grouping, the model recovered the underlying structure without ever seeing the labels.


In [ ]:
from sklearn.manifold import TSNE

# t-SNE needs a flat 2-D input; NaNs are filled with 0 (the series are standardized,
# so 0 = the feature mean, a neutral filler) purely for the projection, not for training
flat = np.nan_to_num(cluster_ds["test_X"]).reshape(len(cluster_ds["test_X"]), -1)
# t-SNE preserves local neighborhood structure, so it usually separates cluster
# structure more clearly than a linear projection like PCA for this kind of check
emb = TSNE(
    n_components=2, random_state=RANDOM_SEED, init="pca", learning_rate="auto"
).fit_transform(flat)

# One shared embedding, two colorings: classes the data was generated with (left) vs.
# the clusters CRLI actually assigned (right) — mismatch shows what the model missed
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, labels, title in [
    (axes[0], cluster_ds["test_y"], "ground-truth classes"),
    (axes[1], cluster_pred, "CRLI cluster assignments"),
]:
    for c in np.unique(labels):
        sel = labels == c
        ax.scatter(emb[sel, 0], emb[sel, 1], s=25, alpha=0.75, label=f"group {c}")
    ax.set_title(title)
    ax.legend()
fig.suptitle("t-SNE projection of the partially-observed test sequences")
plt.tight_layout()
plt.show()

### I.3.5 Anomaly Detection — SAITS with reconstruction-error scoring

The anomaly-detection setup in PyPOTS works as follows:

- **Training**: the model learns to reconstruct _normal_ partially-observed sequences;
- **Scoring**: points with high reconstruction error are flagged as anomalies;
- **Threshold**: the decision threshold is calibrated from the training data at the given `anomaly_rate`.

For evaluation, BenchPOTS can synthesize anomalies into the random-walk data with ground-truth point labels (`test_anomaly_y`).


In [ ]:
# SAITS used for anomaly detection: its attention reconstruction error doubles as
# an anomaly score, so no separate detector architecture is needed
from pypots.anomaly_detection import SAITS as SAITS_AD

# Re-seed before this demo too, so every task section is independently reproducible.
set_random_seed(RANDOM_SEED)

anomaly_rate = 0.05  # 5% of samples get synthetic anomalies — deliberately rare, like real anomalies
anomaly_ds = preprocess_random_walk(
    n_steps=48,
    n_features=5,
    n_classes=2,
    n_samples_each_class=200,
    missing_rate=0.1,
    anomaly_rate=anomaly_rate,
    task_type="anomaly_detection",  # adds per-point anomaly labels (test_anomaly_y) to the returned dict
    random_state=RANDOM_SEED,
)

# Same small capacity as the imputation demo: enough for a tutorial run, and it keeps
# the efficiency comparison in Section I.4 apples-to-apples across tasks
saits_ad = SAITS_AD(
    n_steps=anomaly_ds["n_steps"],
    n_features=anomaly_ds["n_features"],
    anomaly_rate=anomaly_rate,  # tells the model how aggressively to threshold its anomaly scores
    n_layers=1,
    d_model=64,
    n_heads=4,
    d_k=16,
    d_v=16,
    d_ffn=128,
    dropout=0.1,
    batch_size=32,
    epochs=10,
    patience=3,
    num_workers=0,
    device=None,
    saving_path="tutorial_results/anomaly_detection/saits",
    model_saving_strategy="best",  # keep the best-val checkpoint, not the last epoch
)

t0 = time.time()
saits_ad.fit(
    {"X": anomaly_ds["train_X"]}
)  # self-supervised: only X, no anomaly labels needed at train time
saits_ad_train_seconds = time.time() - t0

anomaly_pred = saits_ad.predict({"X": anomaly_ds["test_X"]})[
    "anomaly_detection"
]  # per-point binary labels

# point-level (not sample-level) precision/recall/F1 — the unit of interest is each
# time point, since a sample can be mostly normal with a few anomalous steps
from pypots.nn.functional import calc_precision_recall_f1

y_true_points = anomaly_ds["test_anomaly_y"].reshape(
    -1
)  # flatten to 1-D over all (sample, step) points
_p, _r, _f1 = calc_precision_recall_f1(anomaly_pred.astype(float), y_true_points)
ad_metrics = {"precision": float(_p), "recall": float(_r), "F1": float(_f1)}
print("SAITS anomaly detection:", {k: round(v, 4) for k, v in ad_metrics.items()})
log_experiment(
    "anomaly_detection",
    "SAITS",
    saits_ad,
    saits_ad_train_seconds,
    metrics={k: round(v, 4) for k, v in ad_metrics.items()},
)

Point-level precision and recall tell you _how many_ alarms are correct, but not _whether they fire in the right place_. The two panels check this:

- **Timeline**: overlays the true anomaly points (red) and the model's alarms (orange rings) on the raw series — good detection looks like orange rings centered on red dots, while rings on flat regions are false alarms;
- **Confusion matrix**: gives the same summary in numbers.


In [ ]:
true_anomaly_points = anomaly_ds[
    "test_anomaly_y"
]  # [n_samples, n_steps] — per-point ground truth
predicted_anomaly_points = anomaly_pred.reshape(true_anomaly_points.shape)

sample_id = int(
    true_anomaly_points.sum(axis=1).argmax()
)  # sample with most true anomalies — the richest case to inspect
feature_id = 0
series = np.nan_to_num(
    anomaly_ds["test_X_ori"][sample_id, :, feature_id]
)  # complete (unmasked) series, so the curve has no gaps
true_indices = np.where(true_anomaly_points[sample_id])[0]
predicted_indices = np.where(predicted_anomaly_points[sample_id])[0]

fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2.2, 1]})
axes[0].plot(series, "-", color="steelblue", label="series (feature 0)")
# true anomalies as solid red dots; predicted alarms as hollow orange rings drawn on
# top, so a hit shows as a red dot inside an orange ring and a miss stays plain red
axes[0].scatter(
    true_indices, series[true_indices], color="red", zorder=5, label="true anomaly"
)
axes[0].scatter(
    predicted_indices,
    series[predicted_indices],
    facecolors="none",
    edgecolors="orange",
    s=140,
    linewidths=2,
    zorder=6,
    label="predicted anomaly",
)
axes[0].set_title(f"Anomaly timeline — test sample {sample_id}")
axes[0].legend(fontsize=8)

# point-level confusion matrix over ALL test points: one sample can look good while
# aggregate performance is poor, so show both views side by side
confusion = confusion_matrix(y_true_points, anomaly_pred)
ConfusionMatrixDisplay(confusion, display_labels=["normal", "anomaly"]).plot(
    ax=axes[1], colorbar=False
)
axes[1].set_title("point-level confusion matrix")
plt.tight_layout()
plt.show()

### I.4.1 Metric selection cheat-sheet — the PyPOTS-native way

PyPOTS ships metric helpers per task in `pypots.nn.functional`, so evaluation stays consistent across models instead of being re-implemented per experiment. This notebook uses them throughout:

| Task              | PyPOTS-native helper                       | What it returns             | Notes                                                                       |
| ----------------- | ------------------------------------------ | --------------------------- | --------------------------------------------------------------------------- |
| Imputation        | `calc_mae` / `calc_mse` / `calc_mre`       | scalar                      | evaluated **only on artificially-masked positions** via the indicating mask |
| Forecasting       | `calc_mae` / `calc_mse`                    | scalar                      | computed against the held-out future window `X_pred`                        |
| Classification    | `calc_binary_classification_metrics`       | dict (ROC-AUC, PR-AUC, …)   | PR-AUC is the more informative one under class imbalance                    |
| Clustering        | `calc_external_cluster_validation_metrics` | dict (RI, ARI, NMI, purity) | **label-based**; use when ground truth exists                               |
| Clustering        | `calc_internal_cluster_validation_metrics` | dict (silhouette, CHS, DBS) | **label-free**; the honest choice for real unlabeled POTS                   |
| Anomaly detection | `calc_precision_recall_f1`                 | (P, R, F1)                  | threshold-dependent — always report the threshold                           |

**Metric abbreviations, spelled out.** These short names appear throughout the notebook and the PyPOTS API:

| Abbreviation | Full name                             | How to read it                                                                           |
| ------------ | ------------------------------------- | ---------------------------------------------------------------------------------------- |
| MAE          | Mean Absolute Error                   | average distance between prediction and truth; same unit as the data, robust to outliers |
| MSE          | Mean Squared Error                    | like MAE but squares the errors, so large mistakes are punished more                     |
| MRE          | Mean Relative Error                   | error as a fraction of the true value; useful when features have very different scales   |
| ROC-AUC      | Area Under the ROC curve              | ranking quality over all thresholds; 0.5 = random, 1.0 = perfect                         |
| PR-AUC       | Area Under the Precision–Recall curve | like ROC-AUC but honest under class imbalance                                            |
| RI           | Rand Index                            | fraction of sample pairs on which two clusterings agree; chance level is _not_ 0         |
| ARI          | Adjusted Rand Index                   | RI corrected for chance; 0 = random, 1 = perfect agreement                               |
| NMI          | Normalized Mutual Information         | information shared between clusters and labels; 0 = none, 1 = identical                  |
| CHS          | Calinski–Harabasz Score               | between-cluster vs within-cluster variance; higher = better separated                    |
| DBS          | Davies–Bouldin Score                  | average cluster similarity; **lower** = better separated                                 |

> The anomaly-detection demo below uses the PyPOTS-native `calc_precision_recall_f1` rather than scikit-learn, keeping the whole pipeline inside the ecosystem.


The table above tells you what each metric is; a concrete example shows how it _behaves_.

- **What we build:** one true sequence and two candidate predictions — one good, one poor on purpose — scored with the same PyPOTS helpers used throughout this notebook.
- **What to watch:** how the numbers match what the plot already tells you.


In [ ]:
from pypots.nn.functional import calc_mae, calc_mse, calc_mre

# one true sequence: a smooth sine wave — easy to reason about visually
time_axis = np.linspace(0, 4 * np.pi, 50)
y_true = np.sin(time_axis)

# two candidate predictions of that sequence
y_good = y_true + np.random.default_rng(RANDOM_SEED).normal(
    0, 0.05, size=time_axis.shape
)  # small noise: a good prediction
y_poor = y_true + 0.4  # constant offset: systematically biased

# plot first — always look before you trust a number
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(time_axis, y_true, "o-", color="gray", alpha=0.6, label="ground truth")
ax.plot(
    time_axis,
    y_good,
    "s--",
    color="steelblue",
    markersize=4,
    label="good prediction (small noise)",
)
ax.plot(
    time_axis,
    y_poor,
    "^--",
    color="darkorange",
    markersize=4,
    label="poor prediction (+0.4 bias)",
)
ax.set_title("One true sequence, two predictions")
ax.legend()
plt.show()

# now compute the metrics. The helpers take (prediction, target) — no mask needed here
# because every position is observed. Larger values = worse, for all three.
rows = {
    "good (small noise)": [
        calc_mae(y_good, y_true),
        calc_mse(y_good, y_true),
        calc_mre(y_good, y_true),
    ],
    "poor (+0.4 bias)": [
        calc_mae(y_poor, y_true),
        calc_mse(y_poor, y_true),
        calc_mre(y_poor, y_true),
    ],
}
metric_demo = pd.DataFrame(rows, index=["MAE", "MSE", "MRE"]).T
print(metric_demo.round(4).to_string())
print()
print(
    "MSE punishes the biased prediction more than MAE does, because it squares each error;"
)
print("MRE expresses the same error as a fraction of the true value (scale-free).")

### I.4.2 Imputation error analysis

The global MAE averages away per-feature structure, so we break it down in two ways:

- **Per-feature bar chart** — shows which features are hard to reconstruct;
- **Single-curve view** — focuses on one hard feature for one patient, overlaying the observed input, the imputed points, and the ground truth.

One subtlety when reading these two figures together: the per-feature MAE is averaged over _all_ test patients, while the curve shows only _one_ patient. On a dataset this sparse (~80% missing), a given patient may have no observations at all on the hardest feature — and choosing such a patient for the curve would give an empty flat line that shows nothing. So below we deliberately choose a feature that is both hard _and_ well-observed, and a patient who actually has observations on it.

How to read the curve:

- **Hollow blue points (observed):** they sit exactly on the ground-truth line, because they _are_ true values that the model was shown;
- **Orange squares vs red crosses:** the imputed points compared with the true values at the same masked positions — the gap between them is the imputation error for that point.

The model here was trained for only 10 epochs, so the gap is visible; this is the honest look of a lightly-trained model on its hardest feature, not a polished result.


In [ ]:
# Per-feature MAE on the artificially-masked positions only — never on naturally
# missing values, which have no ground truth to compare against
errors = np.abs(saits_imputation - truth)  # [n_samples, n_steps, n_features]
per_feature_mae = []
for feature_idx in range(n_features):
    feature_mask = mask[:, :, feature_idx]  # masked positions on this one feature
    feature_errors = errors[:, :, feature_idx]
    per_feature_mae.append(
        feature_errors[feature_mask].mean() if feature_mask.any() else np.nan
    )  # NaN if a feature happened to get no masks

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(n_features), per_feature_mae)
ax.set_xlabel("feature index")
ax.set_ylabel("MAE")
ax.set_title(
    "SAITS imputation error per feature"
)  # uneven bars hint at which features are harder to reconstruct
plt.show()

# One reconstructed feature curve for a sample with masked points.
# Pick the feature with the most imputed points overall (a hard, well-represented feature),
# then the sample that has the most observed context on it — so the curve actually shows
# observed points, imputed points, and ground truth together instead of an empty flat line.
feat_idx = int(
    np.argmax(mask.sum(axis=(0, 1)))
)  # the feature with the most masked points overall
obs_cnt = (~np.isnan(test_set["X"][:, :, feat_idx])).sum(axis=1)
imp_cnt = mask[:, :, feat_idx].sum(axis=1)
candidates = np.where((obs_cnt > 0) & (imp_cnt > 0))[
    0
]  # need both: observed context AND masked points to show
sample_idx = int(candidates[np.argmax(obs_cnt[candidates])])

obs = test_set["X"][sample_idx, :, feat_idx]  # model input (has NaNs)
imp = saits_imputation[sample_idx, :, feat_idx]  # model output (complete)
ori = truth[sample_idx, :, feat_idx]  # ground truth
ind = mask[sample_idx, :, feat_idx]  # which positions were artificially masked

fig, ax = plt.subplots(figsize=(10, 4))
# ground truth as a thin full line; observed points sit exactly on it (they ARE the same
# values), so we draw them larger and hollow to keep both visible where they overlap
ax.plot(ori, "-", color="gray", alpha=0.6, linewidth=1.2, label="ground truth")
ax.plot(
    np.where(~np.isnan(obs))[0],
    obs[~np.isnan(obs)],
    "o",
    markersize=7,
    markerfacecolor="white",
    markeredgecolor="steelblue",
    markeredgewidth=1.5,
    linestyle="none",
    label="observed input",
)
ax.plot(
    np.where(ind)[0],
    imp[ind],
    "s",
    markersize=11,
    color="darkorange",
    label="SAITS imputation",
)
# also mark the true value at the imputed positions so the gap is visible
ax.plot(
    np.where(ind)[0],
    ori[ind],
    "x",
    markersize=11,
    color="red",
    markeredgewidth=2.5,
    linestyle="none",
    label="true value (masked)",
)
ax.set_title(f"Sample {sample_idx}, feature {feat_idx}: imputed vs. ground truth")
ax.legend()
plt.show()

The figure shows the same errors in two views:

- **Scatter (imputed vs true values on the masked positions):** points close to the red diagonal mean unbiased reconstruction, while a curved or flattened cloud means the model regresses toward the mean (a classic failure of MSE-trained imputers);
- **Residual histogram:** the same errors as a distribution — you want it centered at zero and roughly symmetric; a shifted or skewed histogram indicates systematic bias.


In [ ]:
# Only artificially-masked positions enter this view: they are the only ones with
# both an imputed value and a ground-truth value to compare
true_values = truth[mask]  # ground truth at the masked positions
imputed_values = saits_imputation[mask]  # model output at the same positions
random_generator = np.random.default_rng(RANDOM_SEED)
# subsample for a readable scatter — plotting every masked point would overplot into a blob
sample_indices = random_generator.choice(
    len(true_values), size=min(5000, len(true_values)), replace=False
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(
    true_values[sample_indices], imputed_values[sample_indices], s=4, alpha=0.3
)
lims = [
    min(true_values.min(), imputed_values.min()),
    max(true_values.max(), imputed_values.max()),
]
# y = x diagonal = perfect imputation; points above/below it are over/under-estimates
axes[0].plot(lims, lims, "r--", label="perfect imputation")
axes[0].set_xlabel("ground truth")
axes[0].set_ylabel("imputed value")
axes[0].set_title("Imputed vs. true values (masked positions)")
axes[0].legend()

# residuals expose what MAE alone hides: bias (mean ≠ 0) and heavy tails
residuals = imputed_values - true_values
axes[1].hist(residuals, bins=60, color="steelblue")
axes[1].axvline(0, color="r", ls="--")  # zero-error reference line
axes[1].set_xlabel("imputation residual (imputed − true)")
axes[1].set_ylabel("count")
axes[1].set_title(
    f"Residual distribution (mean = {residuals.mean():.3f}, std = {residuals.std():.3f})"
)
plt.tight_layout()
plt.show()

### I.4.3 Robustness under increasing missingness + robustness–efficiency tradeoff

Two panels, two questions:

- **Robustness curve (left):** "how does the same trained model degrade as we hide more of its input?" — the x-axis is the _extra_ MCAR rate injected at test time, the y-axis is the MAE on the newly-masked positions, and the slope is the robustness;
- **Efficiency scatter (right):** training time against model size for every model in this tutorial — a model that is 1% more accurate but 10× slower or larger may not be the right choice in practice.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# robustness curve: MAE as extra missingness is injected at test time — the slope
# shows how gracefully the model degrades when observation gets sparser
axes[0].plot(robustness_df["missing_rate"], robustness_df["MAE"], "o-")
axes[0].set_xlabel("additional missing rate injected at test time")
axes[0].set_ylabel("MAE (masked positions)")
axes[0].set_title("SAITS robustness curve")

# efficiency view: every model we ran, one point each — training cost vs. model size
log_df = pd.DataFrame(experiment_log)
for task, grp in log_df.groupby(
    "task"
):  # color by task so cross-task cost differences are visible
    axes[1].scatter(grp["train_seconds"], grp["n_params"], s=80, label=task)
    for _, row in grp.iterrows():
        axes[1].annotate(
            row["model"],
            (row["train_seconds"], row["n_params"]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
        )
axes[1].set_xscale("log")  # log-log: times and param counts span orders of magnitude
axes[1].set_yscale("log")
axes[1].set_xlabel("training wall-clock time (s)")
axes[1].set_ylabel("# trainable parameters")
axes[1].set_title("Efficiency view of the models we ran")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

log_df  # the underlying table, for the report

### I.4.4 Persisting artifacts

`pypots.data.saving.pickle_dump` lets you save everything a report needs in one file — predictions, metrics, and the experiment log.


In [ ]:
from pypots.data.saving import pickle_dump

# Persist everything needed to reproduce the report without re-running any training:
# metrics, the imputation array behind the error-analysis plots, the robustness
# curve, the seed, and the exact library versions
artifacts = {
    "experiment_log": experiment_log,  # metrics + train time + param count for every run
    "saits_imputation_test": saits_imputation,  # the imputed test set, for offline error analysis
    "robustness_curve": robustness_curve,  # missing-rate → MAE pairs from the robustness experiment
    "random_seed": RANDOM_SEED,  # the one knob that reproduces every split/mask/init
    "versions": {
        "pypots": pypots.__version__,
        "benchpots": benchpots.__version__,
    },  # results can shift across versions
}
pickle_dump(artifacts, "tutorial_results/part1_artifacts.pkl")
print("Saved → tutorial_results/part1_artifacts.pkl")

Saving is only useful if a reloaded model behaves identically to the one you trained. The check below reloads the saved SAITS checkpoint into a _fresh_ model instance and confirms that it reproduces the exact same imputations — no retraining, no drift. This is the property that lets you share a checkpoint, deploy it, or pick up an experiment later.


In [ ]:
import glob

# locate the checkpoint that saving_path + model_saving_strategy="best" wrote for us
checkpoints = sorted(glob.glob("tutorial_results/imputation/saits/*/SAITS.pypots"))
checkpoint_path = checkpoints[-1]  # the most recent run
print("Reloading checkpoint:", checkpoint_path)

# a brand-new model instance with the same architecture, then load the trained weights.
# No .fit() call — we want to prove the checkpoint alone restores the model.
saits_reloaded = SAITS(
    n_steps=n_steps,
    n_features=n_features,
    n_layers=2,
    d_model=64,
    n_heads=4,
    d_k=16,
    d_v=16,
    d_ffn=128,
    dropout=0.1,
    batch_size=32,
    epochs=10,
    patience=3,
    device=None,
    verbose=False,
)
saits_reloaded.load(checkpoint_path)

# predict again with the reloaded model and compare to the original imputations
reloaded_imputation = saits_reloaded.predict(test_set)["imputation"]
max_difference = np.abs(reloaded_imputation - saits_imputation).max()
print(
    f"Max absolute difference between original and reloaded predictions: {max_difference:.2e}"
)
print(
    "Predictions are identical:",
    np.allclose(reloaded_imputation, saits_imputation, atol=1e-5),
)

### I.4.5 Reproducibility — the PyPOTS-native mechanisms

Instead of a generic checklist, here is _what PyPOTS already gives you_ and how this notebook uses each mechanism. In this ecosystem, reproducibility comes from the tooling, not only from careful habits:

1. **One seed, everywhere** — `pypots.utils.random.set_random_seed(RANDOM_SEED)` (§0) seeds Python, NumPy _and_ PyTorch in one call. Record the seed with your results (we store it in the artifacts pickle).
2. **Pinned versions** — `pypots / benchpots / pygrinder / tsdb` versions are printed in §0 and saved into the artifacts; pin them in a requirements file for papers.
3. **Automatic experiment logging** — every `saving_path` we set produces a timestamped run directory with the **model checkpoint** _and_ a **TensorBoard log**. Inspect all runs with `tensorboard --logdir tutorial_results/` — no manual bookkeeping.
4. **Best-model checkpointing** — `model_saving_strategy="best"` + `patience` mean the saved model is the early-stopping optimum, so reloading it reproduces the reported metrics exactly.
5. **Documented missingness** — state both the _natural_ missing rate and the _artificial_ evaluation missingness (`pattern` + `rate`, mechanism MCAR/MAR/MNAR). This is the part most papers get wrong.
6. **Ground truth stays separated** — `X_ori` never enters model inputs; imputation metrics are computed only on the indicating mask (§I.2.5).
7. **Unified optimizer/loss** — models default to `pypots.optim.Adam` and task-appropriate losses, so baselines are comparable out of the box; override only when you have a reason.
8. **Artifact persistence** — `pypots.data.saving.pickle_dump` snapshots metrics, predictions, and the experiment log (§I.4.5), so every figure and table is regenerable.

✅ **If you follow all eight, another person can re-run your pipeline and get the same numbers.**


---

## Summary and Pointers

You have built a complete POTS pipeline:

- **I.2** — loaded and standardized PhysioNet-2012 with BenchPOTS, and simulated MCAR/MAR/MNAR/block missingness with PyGrinder;
- **I.3** — trained models across **all five tasks** through one consistent `fit/predict/impute` API (several imputers, a forecaster, an end-to-end classifier, a clusterer, and an anomaly detector), and measured robustness under increasing missing rates;
- **I.4** — selected task-appropriate metrics, analyzed errors visually, and produced persisted, report-ready artifacts under an explicit reproducibility checklist.
